Load the data

In [1]:
#  Loading the data with all 200 features (for 200x200 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\FC1.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['FC1']  # Extract the FC1 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (200, 200, N_subjects)

fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)
fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 200, 200)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 200, 200)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 200, 200)

# Expected shape is (Batch size, channels, height, width)

(200, 200, 72)
(72, 1, 200, 200)
Data shape: torch.Size([72, 1, 200, 200])


In [ ]:
def check_zero_mean_unit_range(data, atol=1e-5):
    if hasattr(data, "detach"):
        arr = data.detach().cpu().numpy()
    else:
        arr = np.asarray(data)

    mean_val = arr.mean()
    min_val = arr.min()
    max_val = arr.max()

    is_zero_mean = np.isclose(mean_val, 0.0, atol=atol)
    has_minus_one = np.isclose(min_val, -1.0, atol=atol)
    has_plus_one = np.isclose(max_val, 1.0, atol=atol)

    print(f"Doing checks for zero mean and unit range:")
    print(f"mean: {mean_val:.6f}")
    print(f"standard deviation: {arr.std():.6f}")
    print(f"min:  {min_val:.6f}")
    print(f"max:  {max_val:.6f}")
    print(f"zero mean: {is_zero_mean}")
    print(f"min == -1: {has_minus_one}")
    print(f"max ==  1: {has_plus_one}")

    return is_zero_mean and has_minus_one and has_plus_one

# Example:
# check_zero_mean_unit_range(X)

def z_normalize(data, eps=1e-8):
    if hasattr(data, "detach"):
        mean = data.mean(dim=(1, 2, 3), keepdim=True)
        std = data.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp_min(eps)
        return (data - mean) / std

    arr = np.asarray(data)
    mean = arr.mean(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = arr.std(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = np.maximum(std, eps)
    return (arr - mean) / std


In [8]:
check_zero_mean_unit_range(X)
z_normalized_X = z_normalize(X)
check_zero_mean_unit_range(z_normalized_X)

Doing checks for zero mean and unit range:
mean: 0.008007
min:  -0.825187
max:  1.000000
zero mean: False
min == -1: False
max ==  1: True
Doing checks for zero mean and unit range:
mean: -0.000000
min:  -2.785055
max:  3.682692
zero mean: True
min == -1: False
max ==  1: False


False

In [1]:
import pandas as pd

def read_csv_and_column_min_max(csv_path):
    df = pd.read_csv(csv_path)
    min_max = pd.DataFrame({
        "min": df.min(numeric_only=True),
        "max": df.max(numeric_only=True)
    })
    return df, min_max

# Example:
# df, column_stats = read_csv_and_column_min_max("your_file.csv")
# print(column_stats)

In [3]:
csv_path = "C:\Mats og Odd Arne\Prosjektoppgave\ISC_data\Beh.csv"

df, column_stats = read_csv_and_column_min_max(csv_path)
for value in  column_stats.itertuples():
    print(f"{value.Index}: min={value.min}, max={value.max}")

Subject: min=11001.0, max=12334.0
Group: min=1.0, max=2.0
Age: min=19.0, max=82.0
Order: min=1.0, max=141.0
Sex: min=1.0, max=2.0
Relationshipstatus: min=1.0, max=5.0
Avg_Sleep: min=4.0, max=10.5
EnglishYearsSpeaking: min=0.0, max=43.0
YearsEducation: min=0.0, max=29.0
PsychDiagnosis: min=0.0, max=1.0
Medicine: min=0.0, max=1.0
NeuroImpairment: min=0.0, max=1.0
EV_base: min=1.0, max=9.0
EV_neu: min=1.0, max=9.0
EV_neg: min=1.0, max=9.0
ER_base: min=1.0, max=9.0
ER_neu: min=1.0, max=9.0
ER_neg: min=1.0, max=9.0
dEV_neu: min=-4.0, max=6.0
dEV_neg: min=-8.0, max=6.0
dER_neu: min=-3.0, max=5.0
dER_neg: min=-4.0, max=8.0
Q1_DASSDepression: min=0.0, max=18.0
Q1_DASSAnxiety: min=0.0, max=14.0
Q1_DASSStress: min=0.0, max=19.0
Q2_DERSScore: min=47.0, max=119.0
Q2_DERSNONACCEPT: min=6.0, max=23.0
Q2_DERSGOALS: min=5.0, max=25.0
Q2_DERSIMPULSE: min=6.0, max=22.0
Q2_DERSAWARENESS: min=6.0, max=28.0
Q2_DERSSTRATEGIES: min=8.0, max=31.0
Q2_DERSCLARITY: min=5.0, max=21.0
Q3_GHQ28Score: min=31.0, max=

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\3478438818.py:1: SyntaxWarning: invalid escape sequence '\M'
  csv_path = "C:\Mats og Odd Arne\Prosjektoppgave\ISC_data\Beh.csv"
